## Pseudobulking in R (alt to Python, later implementation)

In [ ]:
suppressPackageStartupMessages({
    library(Seurat)
    library(SeuratDisk)
    library(muscat)
    library(Matrix)
    library(data.table)
    library(dplyr)
    library(readr)
    library(edgeR)
    library(zellkonverter)
    library(SingleCellExperiment)
    library(SummarizedExperiment)
    library(ggplot2)
    library(stringr)
    library(tidyr)
})

In [ ]:
to_logical <- function(x){
  if (is.logical(x)) return(x)
  y <- tolower(as.character(x))
  out <- ifelse(y %in% c("true","1","t","yes"), TRUE,
                ifelse(y %in% c("false","0","f","no"), FALSE, NA))
  out[is.na(out)] <- FALSE
  as.logical(out)
}

make_filter <- function(beta, panleuko, Tcell, CD8pos, cytoCD8, GLP1Rpos){
  immune_pos <- (panleuko | Tcell | CD8pos | cytoCD8)
  if (beta && !immune_pos && GLP1Rpos) return("beta_pos_immune_neg_GLP1Rpos")
  if (beta && GLP1Rpos)               return("beta_pos_any_immune_GLP1Rpos")
  if (!beta && immune_pos && GLP1Rpos) return("immune_pos_GLP1Rpos_beta_neg")
  if (beta && !immune_pos)             return("beta_pos_immune_neg")
  if (beta)                            return("beta_pos_any_immune")
  return(NA_character_)
}

In [ ]:
seurat_path <- "data/processed/seurat_allsamples_allspots.rds"
h5ad_path   <- "data/processed/lueven_TRUE_R.h5ad"
meta_dir    <- "results/intermediate/pseudobulk_legacy"
out_counts  <- "results/intermediate/pseudobulk_merged_R/pseudobulk_counts_all.csv"
out_cpm     <- "results/intermediate/pseudobulk_merged_R/pseudobulk_CPM_all.csv"
out_meta    <- "results/intermediate/pseudobulk_merged_R/pseudobulk_metadata_all.csv"

group_map <- c(
  FFPE315="untreated_17w", FFPE319="untreated_17w", FFPE320="untreated_17w",
  FFPE317="mono_aCD3_17w", FFPE318="mono_aCD3_17w", FFPE322="mono_aCD3_17w",
  FFPE801="mono_E2GLP1_17w", FFPE802="mono_E2GLP1_17w", FFPE803="mono_E2GLP1_17w",
  FFPE316="combo_aCD3_E2GLP1_17w", FFPE323="combo_aCD3_E2GLP1_17w",
  Mouse1="untreated_12w", Mouse2="untreated_12w", W12F="untreated_12w"
)

BOOL_COLS <- c("beta","panleuko","Tcell","CD8pos","cytoCD8","GLP1Rpos")

In [ ]:
seurat_obj <- readRDS(seurat_path)
sce_all    <- reticulate::import("anndata", delay_load = TRUE)
sce <- zellkonverter::readH5AD(h5ad_path)
counts_matrix <- assay(sce, "X")

all_cells   <- colnames(counts_matrix)
# donor_names <- unique(seurat_obj@meta.data$donor)
donor_names <- c('FFPE315','FFPE316','FFPE317','FFPE318','FFPE319','FFPE320','FFPE322','FFPE323','FFPE801','FFPE802','FFPE803','Mouse1','Mouse2','W12F')

all_counts <- list()
all_meta   <- list()

for (dn in donor_names) {
    donor_name <- dn
    
    # Subset the seurat object 
    donor_layer <- subset(seurat_obj, subset = donor == donor_name)
    cells_layer <- grep(paste0("^", donor_name), all_cells, value = TRUE) 
    counts_layer <- counts_matrix[, cells_layer]
    
    donor_name <- dn   # make sure this matches what you used
    meta_csv   <- file.path(meta_dir, paste0(donor_name, "_otsu_complete_metadata.csv"))
    
    # ---- load metadata booleans (beta, panleuko, Tcell, CD8pos, cytoCD8, GLP1Rpos) ----
    meta <- read.csv(meta_csv)
    stopifnot(all(c("spot","laure_region_id","beta","panleuko","Tcell","CD8pos","cytoCD8","GLP1Rpos") %in% names(meta)))
    
    # coerce True/False/0/1 → logical
    to_logical <- function(x) {
      if (is.logical(x)) return(x)
      y <- as.character(x)
      y <- ifelse(tolower(y) %in% c("true","1","t","yes"), TRUE,
                  ifelse(tolower(y) %in% c("false","0","f","no"), FALSE, NA))
      y[is.na(y)] <- FALSE
      as.logical(y)
    }
    
    for (c in c("beta","panleuko","Tcell","CD8pos","cytoCD8","GLP1Rpos")) 
        meta[[c]] <- to_logical(meta[[c]])
    
    # -- keep only spots present in counts_layer (1:1 match, no harmonization) --
    meta_in_counts <- meta %>% filter(spot %in% colnames(counts_layer))
    if (nrow(meta_in_counts) == 0) stop("No overlap between CSV 'spot' and counts_layer column names.")
    meta_in_counts$col <- meta_in_counts$spot
    
    # -- restrict to labeled ROIs --
    meta_in_counts <- meta_in_counts %>% filter(!is.na(laure_region_id), laure_region_id != "none")
    
    # -- define filter label from booleans (your recipes) --
    make_filter <- function(beta, panleuko, Tcell, CD8pos, cytoCD8, GLP1Rpos){
      immune_pos <- (panleuko | Tcell | CD8pos | cytoCD8)
      if (beta && !immune_pos && GLP1Rpos) return("beta_pos_immune_neg_GLP1Rpos")
      if (beta && GLP1Rpos)               return("beta_pos_any_immune_GLP1Rpos")
      if (!beta && immune_pos && GLP1Rpos) return("immune_pos_GLP1Rpos_beta_neg")
      if (beta && !immune_pos)             return("beta_pos_immune_neg")
      if (beta)                            return("beta_pos_any_immune")
      return(NA_character_)
    }
    meta_in_counts$filter <- mapply(
      make_filter,
      meta_in_counts$beta, meta_in_counts$panleuko, meta_in_counts$Tcell,
      meta_in_counts$CD8pos, meta_in_counts$cytoCD8, meta_in_counts$GLP1Rpos
    )
    meta_in_counts <- meta_in_counts %>% filter(!is.na(filter))
    
    # -- build SCE and aggregate (SUM per ROI × filter) --
    sce2 <- SingleCellExperiment(list(counts = counts_layer[, meta_in_counts$col, drop = FALSE]))
    colData(sce2) <- DataFrame(
      laure_region_id = meta_in_counts$laure_region_id,
      filter          = meta_in_counts$filter
    )
    
    pb <- muscat::aggregateData(
      sce2,
      assay = "counts",
      by    = c("laure_region_id","filter"),
      fun   = "sum"
    )
    
    suppressPackageStartupMessages({ library(SummarizedExperiment); library(Matrix); library(dplyr); library(readr) })
    
    # --- collapse muscat-style assays (one per ROI) into one matrix ---
    anames <- assayNames(pb)                      # e.g. "W12F_Laure_0_4", "W12F_Laure_1_7", ...
    # combine all assays column-wise; tag columns with "<assay>|<filter>"
    mats <- lapply(anames, function(a) {
      m <- assays(pb)[[a]]                       # genes × filters (your groups)
      colnames(m) <- paste(a, colnames(m), sep="|")
      m
    })
    sum_counts <- do.call(cbind, mats)           # genes × (ROI|filter)  (sparse-friendly)
    
    
    # --- build pseudobulk metadata (one row per column of sum_counts) ---
    pb_keys <- colnames(sum_counts)
    pb_meta <- tibble::tibble(key = pb_keys) |>
      tidyr::separate_wider_delim(key, delim="|",
                                 names = c("roi","filter"),
                                 too_few = "align_start") |>
      dplyr::mutate(
        sample_id       = dn,
        laure_region_id = roi
      )
    
    # ---- add n_spots per pseudobulk (ROI × filter) ----
    nspots <- meta_in_counts |>
      dplyr::count(laure_region_id, filter, name = "n_spots")
    
    pb_meta <- pb_meta |>
      dplyr::left_join(nspots, by = c("laure_region_id","filter"))
    
    # ---- boolean % summaries per pseudobulk (ROI × filter) ----
    bool_summ <- meta_in_counts |>
      dplyr::group_by(laure_region_id, filter) |>
      dplyr::summarise(
        dplyr::across(dplyr::all_of(BOOL_COLS),
                      ~ mean(.x, na.rm = TRUE) * 100,
                      .names = "pct_{col}"),
        .groups = "drop"
      )
    
    pb_meta <- pb_meta |>
      dplyr::left_join(bool_summ, by = c("laure_region_id","filter"))
    
    # ---- total counts per pseudobulk (matches columns of sum_counts) ----
    pb_meta$total_counts <- Matrix::colSums(sum_counts)
    
    # --- wide outputs (rows = pseudobulks) ---
    counts_out <- cbind(
      pb_meta[, c("sample_id","laure_region_id","filter","n_spots","total_counts",
                  paste0("pct_", BOOL_COLS))],
      as.data.frame(t(as.matrix(sum_counts)), check.names = FALSE)
    )
    
    # store for later combination
    all_counts[[dn]] <- counts_out
    all_meta[[dn]]   <- pb_meta

}

In [ ]:
counts_all <- dplyr::bind_rows(all_counts)
meta_all   <- dplyr::bind_rows(all_meta)

# add Condition to both
counts_all <- counts_all |>
  dplyr::mutate(Condition = unname(group_map[as.character(sample_id)]))

meta_all <- meta_all |>
  dplyr::mutate(Condition = unname(group_map[as.character(sample_id)]))

# warn if any sample_id didn't map
missing_ids <- setdiff(unique(as.character(counts_all$sample_id)), names(group_map))
if (length(missing_ids) > 0) {
  warning("No Condition mapping for sample_id: ", paste(missing_ids, collapse = ", "))
}

readr::write_csv(counts_all, out_counts)

meta_cols <- c("sample_id","Condition","laure_region_id","filter","n_spots","total_counts",
               paste0("pct_", BOOL_COLS))

gene_cols <- setdiff(colnames(counts_all), meta_cols)

counts_mat <- t(as.matrix(counts_all[, gene_cols, drop = FALSE]))
storage.mode(counts_mat) <- "numeric"

libs <- Matrix::colSums(counts_mat)
bad  <- !is.finite(libs) | libs <= 0
if (any(bad)) message("Dropping ", sum(bad), " zero-library pseudobulks before CPM.")

counts_pos <- counts_mat[, !bad, drop = FALSE]
meta_pos   <- counts_all[!bad, meta_cols, drop = FALSE]

cpm_pos <- edgeR::cpm(counts_pos, log = FALSE)

cpm_out <- cbind(
  meta_pos,
  as.data.frame(t(cpm_pos), check.names = FALSE)
)

readr::write_csv(cpm_out, out_cpm)
readr::write_csv(counts_all[, meta_cols, drop = FALSE], out_meta)

In [ ]:
# 1) Read the wide counts (rows = pseudobulks, cols = metadata + genes)
counts_out <- read.csv("results/intermediate/pseudobulk_merged_R/pseudobulk_counts_all.csv", check.names = FALSE)

# 2) Split meta vs genes
meta_cols <- c(
  "sample_id","Condition","laure_region_id","filter","n_spots","total_counts",
  "pct_beta","pct_panleuko","pct_Tcell","pct_CD8pos","pct_cytoCD8","pct_GLP1Rpos"
)
stopifnot(all(meta_cols %in% colnames(counts_out)))

pb_meta <- counts_out[, meta_cols, drop = FALSE]
gene_counts <- counts_out[, setdiff(colnames(counts_out), meta_cols), drop = FALSE]

# 3) Rebuild sum_counts (genes x pseudobulks) and set a stable column key
sum_counts <- t(as.matrix(gene_counts))                    # genes x pseudobulks
col_key <- paste(pb_meta$sample_id, pb_meta$laure_region_id, pb_meta$filter, sep="|")
colnames(sum_counts) <- col_key

# 4) Extract GRADE from laure_region_id and drop NA / "Q"
pb_meta$GRADE <- stringr::str_match(pb_meta$laure_region_id, ".*_Laure_([^_]+)_.*")[,2]
keep_grade <- !is.na(pb_meta$GRADE) & pb_meta$GRADE != "Q"

pb_meta <- pb_meta[keep_grade, , drop=FALSE]
sum_counts <- sum_counts[, keep_grade, drop=FALSE]
col_key <- col_key[keep_grade]

# Optional: ordered factor
pb_meta$GRADE <- factor(pb_meta$GRADE, levels = c("0","1","2","3"), ordered = TRUE)

# 5) Apply QC thresholds
min_spots  <- 3
min_counts <- 10000
keep_qc <- (pb_meta$n_spots >= min_spots) & (pb_meta$total_counts >= min_counts)

pb_meta_cut   <- pb_meta[keep_qc, , drop=FALSE]
sum_counts_cut <- sum_counts[, keep_qc, drop=FALSE]
col_key_cut    <- col_key[keep_qc]

# Sanity checks
stopifnot(ncol(sum_counts_cut) == nrow(pb_meta_cut))
stopifnot(identical(colnames(sum_counts_cut), paste(pb_meta_cut$sample_id, pb_meta_cut$laure_region_id, pb_meta_cut$filter, sep="|")))

# (optional) see how many dropped at each step
cat("Dropped by GRADE (NA/Q):", sum(!keep_grade), "\n")
cat("Dropped by QC (spots/counts):", sum(!keep_qc), "\n")
cat("Remaining:", nrow(pb_meta_cut))